## טיב ההתאמה: חי בריבוע ו-R בריבוע

יש לנו התאמה (`m`, `b`) ואת שגיאות המקדמים. השאלה הבאה: **האם ההתאמה בכלל טובה**? יש לזה שתי מדדים נפוצים, שעונים על שאלות שונות:

- **חי בריבוע מצומצם** (reduced chi-squared, $\chi^2_\nu$): האם השאריות תואמות את **שגיאות המדידה הידועות** של הנקודות? דורש שיהיה לנו `sigma_y` לכל נקודה.
- **R בריבוע** ($R^2$): איזה חלק מהפיזור הכולל בנתונים "מוסבר" על ידי המודל, יחסית לפיזור סביב הממוצע הפשוט. לא דורש `sigma_y`.

In [ ]:
import numpy as np
import pandas as pd

g = 9.8
df = pd.read_csv("lab_measurements.csv")
df_clean = df.dropna()
angles = sorted(df_clean["angle_deg"].unique())

x = np.array([np.sin(2*np.radians(a)) for a in angles])
y = np.array([df_clean[df_clean["angle_deg"] == a]["range_measured"].mean() for a in angles])
sigma_y = np.array([
    df_clean[df_clean["angle_deg"] == a]["range_measured"].std(ddof=1) / np.sqrt(len(df_clean[df_clean["angle_deg"] == a]))
    for a in angles
])

def linear_fit(x, y):
    x_bar, y_bar = x.mean(), y.mean()
    m = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar)**2)
    b = y_bar - m * x_bar
    return m, b

m, b = linear_fit(x, y)
y_pred = m*x + b
resid = y - y_pred
print("sigma_y לכל זווית:", np.round(sigma_y, 3))

### חי בריבוע מצומצם

$$\chi^2 = \sum \left(\frac{y_i - \hat y_i}{\sigma_{y_i}}\right)^2 \qquad \chi^2_\nu = \frac{\chi^2}{n - k}$$

כאשר `k=2` (מספר הפרמטרים המותאמים). **פירוש**: $\chi^2_\nu \approx 1$ -- ההתאמה טובה, השאריות בגודל שמתאים לשגיאות המדידה. $\chi^2_\nu \gg 1$ -- המודל לא מסביר את הנתונים היטב (או ששגיאות המדידה מוערכות בחסר). $\chi^2_\nu \ll 1$ -- שגיאות המדידה מוערכות ביתר (או "התאמת יתר").

In [ ]:
n = len(x)
k = 2
dof = n - k

chi2 = np.sum((resid / sigma_y)**2)
reduced_chi2 = chi2 / dof

print(f"chi^2 = {chi2:.3f}")
print(f"chi^2_nu (מצומצם) = {reduced_chi2:.3f}   (dof={dof})")

### R בריבוע

$$R^2 = 1 - \frac{\sum (y_i - \hat y_i)^2}{\sum (y_i - \bar y)^2}$$

המונה: סכום ריבועי השאריות של המודל. המכנה: סכום ריבועי הסטייה מהממוצע הפשוט (המודל ה"טיפשי" ביותר -- קו אופקי בגובה הממוצע). $R^2$ קרוב ל-1 אומר שהמודל מסביר הרבה יותר שונות מאשר סתם ממוצע.

In [ ]:
ss_res = np.sum(resid**2)
ss_tot = np.sum((y - y.mean())**2)
r_squared = 1 - ss_res/ss_tot
print(f"R^2 = {r_squared:.4f}")

### באג נפוץ: chi-squared בלי לחלק ב-sigma, או R² בלי דרגות חופש

1. לחשב `np.sum(resid**2)` ולקרוא לזה "chi-squared" -- זה סכום ריבועי שאריות **גולמי**, ללא נרמול לפי שגיאת המדידה של כל נקודה. שתי נקודות עם אותה שארית אבל `sigma_y` שונה לגמרי צריכות לתרום תרומה שונה ל-$\chi^2$ -- בלי החלוקה, זה אבוד.
2. לבלבל בין $\chi^2$ (הסכום הגולמי) לבין $\chi^2_\nu$ (המצומצם, אחרי חלוקה ב-dof) -- $\chi^2$ גדל עם `n` גם אם ההתאמה מושלמת, ולכן **אי אפשר** לפרש אותו לבד כ"טוב/רע" בלי לדעת גם את `n` ו-`k`.

In [ ]:
chi2_no_norm = np.sum(resid**2)   # "חי בריבוע" בלי חלוקה ב-sigma - זו טעות מושגית
print(f"סכום שאריות בריבוע (לא מנורמל): {chi2_no_norm:.3f}")
print(f"chi^2 אמיתי (מנורמל לפי sigma_y): {chi2:.3f}")
print("שני המספרים לא אמורים להיות זהים - הנרמול הוא בדיוק העניין.")

### נסו בעצמכם

חשבו `chi^2_nu` ו-`R^2` עבור התאמה של `run_id` מול `range_measured` בתוך זווית 30 בלבד (עם `sigma_y` קבוע = סטיית התקן של המדידה הבודדת, לא ה-SEM, כי כאן משווים מדידה בודדת למודל, לא ממוצע קבוצה).

In [ ]:
# sub30 = df_clean[df_clean["angle_deg"] == 30]
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
sub30 = df_clean[df_clean["angle_deg"] == 30]
x30 = sub30["run_id"].to_numpy(dtype=float)
y30 = sub30["range_measured"].to_numpy(dtype=float)
sigma30 = y30.std(ddof=1)   # סטיית תקן של מדידה בודדת (לא SEM)

m30, b30 = linear_fit(x30, y30)
resid30 = y30 - (m30*x30 + b30)
n30 = len(x30)
dof30 = n30 - 2

chi2_30 = np.sum((resid30/sigma30)**2)
reduced_chi2_30 = chi2_30 / dof30
ss_res30 = np.sum(resid30**2)
ss_tot30 = np.sum((y30 - y30.mean())**2)
r2_30 = 1 - ss_res30/ss_tot30

print(f"chi^2_nu = {reduced_chi2_30:.3f}   R^2 = {r2_30:.3f}")
```
שימו לב: ציפינו לשיפוע קרוב ל-0 (v0 לא אמור להשתנות עם run_id), ולכן `R^2` כאן צפוי להיות **נמוך** -- לא כי ההתאמה "גרועה", אלא כי אין בכלל מגמה ליניארית אמיתית להסביר; רוב הפיזור הוא רעש. `R^2` נמוך פה הוא בעצם תוצאה נכונה ומצופה.
`````

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "התאמה נתנה chi^2_nu = 0.3. מה זה עשוי לרמז?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "ששגיאות המדידה (sigma_y) הוערכו ביתר (גדולות מדי) ביחס לפיזור בפועל, או שיש התאמת יתר", "correct": True, "feedback": "נכון - השאריות קטנות משמעותית מכפי שהשגיאות המוצהרות מרמזות."},
            {"answer": "שההתאמה גרועה מאוד ויש לזרוק אותה", "correct": False, "feedback": "לא - chi^2_nu נמוך מ-1 בדרך כלל לא אומר התאמה גרועה."},
            {"answer": "אין שום מידע ב-chi^2_nu שלא כבר קיים ב-R^2", "correct": False, "feedback": "לא נכון - chi^2_nu משתמש במידע נוסף (sigma_y) ש-R^2 לא משתמש בו."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

ארזו chi², chi²_nu ו-R² לתוך `fit_with_errors` (שכתבתם בסעיף 11.7) -- הוסיפו פרמטר אופציונלי `sigma_y=None`; אם `sigma_y` לא ניתן, יש לדלג על חישוב chi² (או להחזיר `None` עבורו) ולחשב רק R². הריצו על נתוני x, y הראשיים עם ה-`sigma_y` שחישבנו.

In [ ]:
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
def fit_with_errors(x, y, sigma_y=None):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m, b = linear_fit(x, y)
    y_pred = m*x + b
    resid = y - y_pred
    n = len(x)
    dof = n - 2

    ss_res = np.sum(resid**2)
    ss_tot = np.sum((y - y.mean())**2)
    r2 = 1 - ss_res/ss_tot

    out = {"m": m, "b": b, "resid": resid, "r_squared": r2}
    if sigma_y is not None:
        chi2 = np.sum((resid/sigma_y)**2)
        out["chi2"] = chi2
        out["reduced_chi2"] = chi2 / dof
    else:
        out["chi2"] = None
        out["reduced_chi2"] = None
    return out

result = fit_with_errors(x, y, sigma_y=sigma_y)
print(result["reduced_chi2"], result["r_squared"])
```
`````